In [33]:
import os
import json
import logging
from datetime import datetime, timezone
from dotenv import load_dotenv
from telethon import TelegramClient

In [27]:
load_dotenv(dotenv_path="../.env", override=True)

API_ID = os.getenv("TG_API_ID")
API_HASH = os.getenv("TG_API_HASH")

# Target Telegram Channels specified in your brief
CHANNELS = ['CheMed123', 'lobelia4cosmetics', 'tikvahpharma']

print(f"Loaded credentials. Target Channels: {CHANNELS}")

Loaded credentials. Target Channels: ['CheMed123', 'lobelia4cosmetics', 'tikvahpharma']


In [28]:
# Setup basic logging directory inside the workspace
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=f"{log_dir}/scraping_{datetime.utcnow().strftime('%Y%m%d')}.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# Also echo to notebook console output
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger('').addHandler(console)

logging.info("Scraping logs subsystem initialized.")

/tmp/ipykernel_77119/2243446145.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  filename=f"{log_dir}/scraping_{datetime.utcnow().strftime('%Y%m%d')}.log",
Scraping logs subsystem initialized.
Scraping logs subsystem initialized.
Scraping logs subsystem initialized.
Scraping logs subsystem initialized.
Scraping logs subsystem initialized.
Scraping logs subsystem initialized.


In [34]:
async def scrape_channel_to_lake(client, channel_username):
    logging.info(f"Starting extraction for channel: {channel_username}")
    today_partition = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    
    msg_output_dir = f"../data/raw/telegram_messages/{today_partition}"
    img_output_dir = f"../data/raw/images/{channel_username}"
    os.makedirs(msg_output_dir, exist_ok=True)
    os.makedirs(img_output_dir, exist_ok=True)
    
    extracted_records = []
    image_download_count = 0
    MAX_IMAGES_PER_CHANNEL = 5 # Prevents your network from bottlenecking
    
    try:
        async for message in client.iter_messages(channel_username, limit=100):
            saved_image_path = None
            
            # Smart Capping: Only download the physical file if it fits within our threshold limit
            if message.photo:
                if image_download_count < MAX_IMAGES_PER_CHANNEL:
                    saved_image_path = f"{img_output_dir}/{message.id}.jpg"
                    await message.download_media(file=saved_image_path)
                    image_download_count += 1
                else:
                    # Flag that it has an image on Telegram, but don't waste bandwidth downloading the file
                    saved_image_path = "SKIPPED_FOR_PERFORMANCE"
            
            extracted_records.append({
                "message_id": message.id,
                "channel_name": channel_username,
                "message_date": message.date.isoformat() if message.date else None,
                "message_text": message.text if message.text else "",
                "has_media": message.photo is not None,
                "image_path": saved_image_path,
                "views": message.views if message.views else 0,
                "forwards": message.forwards if message.forwards else 0
            })
            
        target_json_path = f"{msg_output_dir}/{channel_username}.json"
        with open(target_json_path, 'w', encoding='utf-8') as json_file:
            json.dump(extracted_records, json_file, ensure_ascii=False, indent=4)
            
        logging.info(f"Successfully serialized {len(extracted_records)} posts from {channel_username}. Downloaded {image_download_count} sample images.")
        
    except Exception as e:
        logging.error(f"Error encountered while extracting data from {channel_username}: {str(e)}")

In [35]:
# Initialize and execute within Jupyter's ambient async environment loop safely
client = TelegramClient('../session_notebook', int(API_ID), API_HASH)

async def run_pipeline():
    async with client:
        for channel in CHANNELS:
            await scrape_channel_to_lake(client, channel)

# Fire execution pipeline tracking logic inside notebook instance runtime
await run_pipeline()
print("\n--- Pipeline Completed Successfully ---")

Connecting to 149.154.167.91:443/TcpFull...
Connecting to 149.154.167.91:443/TcpFull...
Connecting to 149.154.167.91:443/TcpFull...
Connecting to 149.154.167.91:443/TcpFull...
Connecting to 149.154.167.91:443/TcpFull...
Connecting to 149.154.167.91:443/TcpFull...
Connection to 149.154.167.91:443/TcpFull complete!
Connection to 149.154.167.91:443/TcpFull complete!
Connection to 149.154.167.91:443/TcpFull complete!
Connection to 149.154.167.91:443/TcpFull complete!
Connection to 149.154.167.91:443/TcpFull complete!
Connection to 149.154.167.91:443/TcpFull complete!
Starting extraction for channel: CheMed123
Starting extraction for channel: CheMed123
Starting extraction for channel: CheMed123
Starting extraction for channel: CheMed123
Starting extraction for channel: CheMed123
Starting extraction for channel: CheMed123
Starting direct file download in chunks of 131072 at 0, stride 131072
Starting direct file download in chunks of 131072 at 0, stride 131072
Starting direct file download in


--- Pipeline Completed Successfully ---
